# Build LGBM features + train LambdaMART ranker (A1)

One-shot pipeline:
1. Clone `fresh-model`
2. pip install requirements
3. Run `scripts/build_lgbm_features.py` — extracts wRRF candidates + 11 features per (query, candidate) from train conversations. Output: `data/lgbm_features.parquet`.
4. Train LightGBM LambdaRank on the parquet, evaluate on held-out session split.
5. Save booster + feature-schema + metadata to Drive for the inference reranker (exp 027) to pick up.

**Default**: 2000 train sessions × ~4 music-turns/session × 100 wRRF candidates = ~800k training rows. LightGBM LambdaRank is fast (~3–5 min on CPU).

**Wall time on A100**: ~30 min total (bulk is feature extraction; LGBM training is CPU-bound and quick).

Output → `/content/drive/MyDrive/recsys2026-lgbm-ranker/lgbm_ranker.zip`.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 3) Install deps. requirements.txt covers bm25s + omegaconf + pandas (needed
# by the mcrs cascade the extractor imports), then lightgbm + pyarrow on top.
!pip install -q -r requirements.txt
!pip install -q lightgbm pyarrow scikit-learn
!python -c "import bm25s, lightgbm, torch, transformers, omegaconf; print('bm25s', bm25s.__version__, 'lightgbm', lightgbm.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) (Optional) HF token for faster downloads of TalkPlay datasets.
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Extract features. ~800k rows from 2000 sessions. Expected ~10–15 min on
# A100 (dense encoder is the bottleneck; wRRF re-runs 3 subs per batch).
N_SESSIONS = 2000
TOPK = 100
!python scripts/build_lgbm_features.py --n-sessions {N_SESSIONS} --topk {TOPK} --out data/lgbm_features.parquet --cache-dir /content/recsys2026-lora-tutorial/experiments/cache
!ls -lh data/lgbm_features.parquet

In [ ]:
# 6) Train LightGBM LambdaRank.
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import ndcg_score

df = pd.read_parquet('data/lgbm_features.parquet')
print(f'total rows: {len(df):,}  positives: {int(df.label.sum()):,}  positive rate: {df.label.mean():.4f}')

# Session-level train/val split (90/10) to avoid intra-session leakage.
all_sessions = sorted(df['session_id'].unique())
rng = np.random.default_rng(42)
perm = rng.permutation(len(all_sessions))
n_val = max(1, int(len(all_sessions) * 0.1))
val_sessions = set([all_sessions[i] for i in perm[:n_val]])
df_train = df[~df['session_id'].isin(val_sessions)].reset_index(drop=True)
df_val = df[df['session_id'].isin(val_sessions)].reset_index(drop=True)
print(f'train: {len(df_train):,} rows over {df_train.query_id.nunique():,} queries')
print(f'val:   {len(df_val):,} rows over {df_val.query_id.nunique():,} queries')

FEATURES = [
    # numeric
    'wrrf_rank', 'cfbpr_score', 'pop_log', 'recency_years',
    'tag_count', 'artist_in_query',
    # categorical (LGBM handles strings via categorical_feature)
    'goal_category', 'goal_specificity',
    'user_age_group', 'user_country', 'user_gender',
]
CAT_COLS = ['goal_category', 'goal_specificity', 'user_age_group', 'user_country', 'user_gender']

# LGBM requires categoricals as int-coded 'category' dtype
for c in CAT_COLS:
    df_train[c] = df_train[c].astype('category')
    # Align val categories to train's levels so unseen values map to NaN (LGBM handles it).
    df_val[c] = pd.Categorical(df_val[c], categories=df_train[c].cat.categories)

# Group sizes (rows per query) — required for LambdaRank.
def group_sizes(d):
    return d.groupby('query_id', sort=False).size().values

train_ds = lgb.Dataset(
    df_train[FEATURES], label=df_train['label'], group=group_sizes(df_train),
    categorical_feature=CAT_COLS,
)
val_ds = lgb.Dataset(
    df_val[FEATURES], label=df_val['label'], group=group_sizes(df_val),
    categorical_feature=CAT_COLS, reference=train_ds,
)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [1, 10, 20],
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbosity': -1,
    'seed': 42,
}
booster = lgb.train(
    params, train_ds,
    num_boost_round=500,
    valid_sets=[val_ds],
    valid_names=['val'],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(50)],
)
print(f'\ntrained — best iter: {booster.best_iteration}  best val ndcg@20: {booster.best_score["val"]["ndcg@20"]:.4f}')

In [ ]:
# 7) Feature-importance + spot-check.
import pandas as pd
imp = pd.DataFrame({
    'feature': booster.feature_name(),
    'gain': booster.feature_importance(importance_type='gain'),
    'split': booster.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)
print('\n=== feature importance ===')
print(imp.to_string(index=False))

In [ ]:
# 8) Save booster + metadata; zip + upload to Drive.
import os, json, shutil
OUT = '/content/lgbm_ranker'
os.makedirs(OUT, exist_ok=True)
booster.save_model(f'{OUT}/booster.txt', num_iteration=booster.best_iteration)
cat_maps = {c: list(df_train[c].cat.categories) for c in CAT_COLS}
meta = {
    'features': FEATURES,
    'categorical_features': CAT_COLS,
    'categorical_levels': cat_maps,
    'best_iteration': booster.best_iteration,
    'best_val_ndcg20': float(booster.best_score['val']['ndcg@20']),
    'trained_on_sessions': int(df_train['session_id'].nunique()),
    'trained_on_rows': int(len(df_train)),
}
with open(f'{OUT}/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

shutil.make_archive('/content/lgbm_ranker', 'zip', OUT)
!ls -lh /content/lgbm_ranker.zip

from google.colab import drive
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-lgbm-ranker'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/lgbm_ranker.zip', dst)
print(f'\nsaved to: {dst}')
!ls -lh {dst}

In [ ]:
# 9) Optional browser download.
from google.colab import files
files.download('/content/lgbm_ranker.zip')

## What next

After this notebook finishes, the ranker zip is in Drive. Exp 027 (the Blind-A ship) will pull it via `colab/Run_027_wrrf_lgbm.ipynb` — same pattern as Run_026 did for the reward model.

**If val ndcg@20 > 0.15** (vs 021's offline 0.1209): strong signal the ranker lifts retrieval. Expected Blind-A lift: +0.03–0.08 nDCG@20 (clears ±0.05 noise floor).

**If val ndcg@20 ≈ 0.12**: marginal — model didn't learn much beyond wRRF's own ordering. Reconsider feature engineering or drop cf-bpr weight.

**If val ndcg@20 < 0.12**: worse than wRRF alone. Likely bug or feature misalignment; investigate before shipping.